# TMDb Data Profile

This notebook documents the data-quality checks performed on the candidate movie list and the TMDb responses. It records the issues found, their effect on movie coverage, and the evidence behind decisions used in the pipeline.

## 1. Candidate ID quality

The supplied CSV contains one column of candidate TMDb movie IDs. The first check accounts for every input row and identifies which distinct IDs can be sent to TMDb.

In [5]:
from pathlib import Path

import pandas as pd

# Supports starting the notebook from the project root or Data Profile.
project_root = Path.cwd()
if project_root.name.lower() == "data profile":
    project_root = project_root.parent

audit_path = project_root / "data" / "validated" / "id_audit.csv"
id_audit = pd.read_csv(audit_path, dtype={"tmdb_id": str}, keep_default_na=False)

status_order = ["missing", "invalid_format", "duplicate", "ready_to_fetch"]
counts = id_audit["status"].value_counts().reindex(status_order, fill_value=0)

assert counts.sum() == len(id_audit), "The audit has an unexpected status."

print(f"Total input rows: {len(id_audit)}")
display(counts.rename("rows").to_frame())

Total input rows: 127


,rows
status,
missing,1
invalid_format,3
duplicate,6
ready_to_fetch,117


In [6]:
issues = id_audit.loc[
    id_audit["status"].ne("ready_to_fetch"),
    ["source_line", "tmdb_id", "status"],
].copy()

# Quotes make the blank value visible as '' in the notebook.
issues["tmdb_id"] = issues["tmdb_id"].map(repr)
display(issues)

,source_line,tmdb_id,status
28,30,'-1',invalid_format
41,43,'603',duplicate
58,60,'278',duplicate
67,69,'',missing
88,90,'1891',duplicate
89,91,'155',duplicate
92,94,'550',duplicate
93,95,'tt0468569',invalid_format
100,102,'680',duplicate
109,111,'N/A',invalid_format


### Finding and decision

Of the 127 input rows, 1 was blank, 3 failed the positive-integer rule,
and 6 were repeat occurrences of IDs already present in the list.
The malformed values were `-1`, `tt0468569`, and `N/A`.

The pipeline kept the first occurrence of each eligible ID. That left
117 distinct IDs to request from TMDb. The six duplicate rows added no
new IDs. The four unusable rows reduced coverage of the supplied list.

Reconciliation: 127 = 1 blank + 3 invalid + 6 duplicates + 117 ready to fetch.

## 2. TMDb response structure

I inspected the cached details and credits for The Dark Knight (TMDb ID 155).
The goal was to see how TMDb represents movie attributes, collection
membership, genres, cast, and crew before deciding how to flatten them.

In [7]:
import json
import sqlite3
from contextlib import closing

cache_path = project_root / "data" / "cache" / "tmdb.sqlite"
movie_id = "155"

if not cache_path.is_file():
    raise FileNotFoundError(f"Run pipeline/fetch.py first. Cache missing at {cache_path}")

with closing(sqlite3.connect(cache_path)) as connection:
    rows = connection.execute(
        """
        SELECT endpoint, payload
        FROM responses
        WHERE movie_id = ?
        """,
        (movie_id,),
    ).fetchall()

responses = {endpoint: json.loads(payload) for endpoint, payload in rows}
assert responses.keys() == {"details", "credits"}, f"Missing response for {movie_id}"

details = responses["details"]
credits = responses["credits"]
assert str(details["id"]) == str(credits["id"]) == movie_id

print(f"Movie: {details['title']} ({movie_id})")
print("Details fields:", list(details))
print("Credits fields:", list(credits))

Movie: The Dark Knight (155)
Details fields: ['adult', 'backdrop_path', 'belongs_to_collection', 'budget', 'genres', 'homepage', 'id', 'imdb_id', 'origin_country', 'original_language', 'original_title', 'overview', 'popularity', 'poster_path', 'production_companies', 'production_countries', 'release_date', 'revenue', 'runtime', 'softcore', 'spoken_languages', 'status', 'tagline', 'title', 'video', 'vote_average', 'vote_count']
Credits fields: ['id', 'cast', 'crew']


In [ ]:
collection = details["belongs_to_collection"]
first_billed_actor = min(credits["cast"], key=lambda actor: actor["order"])
directors = [person for person in credits["crew"] if person["job"] == "Director"]

print("Collection:", collection["id"], collection["name"])
print("Genres:", [genre["name"] for genre in details["genres"]])
print("First-billed actor:", first_billed_actor["name"])
print("First crew entry:", credits["crew"][0]["name"], credits["crew"][0]["job"])
print("Directors:", [person["name"] for person in directors])